In [1]:
import os
import numpy as np
import pandas as pd
import lightkurve as lk

def detrend_lc(lc, window_hours=6):
    window_len = int(window_hours * 180)
    if window_len % 2 == 0: 
        window_len += 1
    return lc.flatten(window_length=window_len, return_trend=True)

def find_flares(flat_lc, sigma_thresh=4.0, min_points=3):
    flux = flat_lc.flux.value
    med = np.median(flux)
    mad = np.median(np.abs(flux - med))
    sigma = 1.4826 * mad
    
    mask = flux > (med + sigma_thresh * sigma)
    idx = []
    i = 0
    while i < len(mask) - min_points:
        if all(mask[i : i + min_points]):
            for j in range(min_points):
                if (i + j) not in idx: 
                    idx.append(i + j)
            i += min_points
        else:
            i += 1
    return np.array(idx), med, sigma

def get_metrics(flat_lc, flare_idx, med, sigma):
    flux = flat_lc.flux.value
    time = flat_lc.time.value
    if len(flare_idx) == 0: 
        return []
        
    groups = np.split(flare_idx, np.where(np.diff(flare_idx) > 15)[0] + 1)
    events = []
    
    for g in groups:
        if len(g) < 3: 
            continue
        p_idx = g[np.argmax(flux[g])]
        t_peak = time[p_idx]
        
        t_start_idx = g[0]
        while t_start_idx > 0 and flux[t_start_idx] > (med + 0.5 * sigma): 
            t_start_idx -= 1
        t_start = time[t_start_idx]
        
        t_stop_idx = g[-1]
        while t_stop_idx < len(flux) - 1 and flux[t_stop_idx] > (med + 0.5 * sigma): 
            t_stop_idx += 1
        t_stop = time[t_stop_idx]
        
        dt_rise = (t_peak - t_start) * 1440.0
        dt_total = (t_stop - t_start) * 1440.0
        
        if dt_total <= 0 or dt_rise <= 0 or dt_rise >= dt_total: 
            continue
            
            # removes low-amplitude noise
        amplitude = flux[p_idx] - med
        if amplitude < 0.01:
            continue
            
        events.append({
            "t_start": t_start,
            "t_peak": t_peak,
            "t_stop": t_stop,
            "rise_min": dt_rise,
            "tot_min": dt_total,
            "ratio": dt_rise / dt_total,
            "amp": amplitude
        })
    return events

/opt/anaconda3/lib/python3.13/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [2]:
# Statistically balanced sample across the M4 convective boundary
targets = [
    # partially conv
    {"tic": 388857263, "spt": "M3V"},
    {"tic": 259962529, "spt": "M2.5V"},
    {"tic": 141829023, "spt": "M3.5V"},
    {"tic": 237554904, "spt": "M3V"},
    {"tic": 402539050, "spt": "M3.5V"},
    {"tic": 285089204, "spt": "M2V"},
    {"tic": 307210830, "spt": "M3V"},
    {"tic": 234349833, "spt": "M3.5V"},
    
    # fully conv
    {"tic": 234295610, "spt": "M4V"},
    {"tic": 177309964, "spt": "M4.5V"},
    {"tic": 441162396, "spt": "M5V"},
    {"tic": 231541244, "spt": "M6V"},
    {"tic": 300418356, "spt": "M4.5V"},
    {"tic": 150410712, "spt": "M5.5V"},
    {"tic": 425933644, "spt": "M4V"},
    {"tic": 124211115, "spt": "M5V"}
]

In [3]:
print("flare extraction pipeline\n")

if os.path.exists("extracted_flares.csv"):
    os.remove("extracted_flares.csv")

for t in targets:
    try:
        print(f"Querying MAST for TIC {t['tic']} ({t['spt']})...")
        search = lk.search_lightcurve(f"TIC {t['tic']}", author="SPOC", exptime=20)
        
        # stars w/o data
        if len(search) == 0:
            print("  --> No 20-second SPOC data found")
            continue
            
        lc = search.download_all().stitch().normalize().remove_nans()
        
        flat_lc, trend_lc = detrend_lc(lc)
        flare_idx, med, sigma = find_flares(flat_lc)
        flares = get_metrics(flat_lc, flare_idx, med, sigma)
        
        if len(flares) > 0:
            df = pd.DataFrame(flares)
            df["tic_id"] = t["tic"]
            df["spectral_type"] = t["spt"]
            
            file_exists = os.path.isfile("extracted_flares.csv")
            df.to_csv("extracted_flares.csv", mode="a", index=False, header=not file_exists)
            print(f"  --> Success: Extracted {len(flares)} qualified flares.")
        else:
            # identify stars that have 0 flares passing the threshold
            print("  --> 0 baseline-crossing flares met amplitude constraints.")
            
    except Exception as e:
        print(f"  --> error processing TIC {t['tic']}: {e}")

print("\ndata in 'extracted_flares.csv'.")

flare extraction pipeline

Querying MAST for TIC 388857263 (M3V)...
  --> Success: Extracted 72 qualified flares.
Querying MAST for TIC 259962529 (M2.5V)...
  --> No 20-second SPOC data found
Querying MAST for TIC 141829023 (M3.5V)...
  --> No 20-second SPOC data found
Querying MAST for TIC 237554904 (M3V)...
  --> No 20-second SPOC data found
Querying MAST for TIC 402539050 (M3.5V)...
  --> No 20-second SPOC data found
Querying MAST for TIC 285089204 (M2V)...
  --> No 20-second SPOC data found
Querying MAST for TIC 307210830 (M3V)...
  --> Success: Extracted 4 qualified flares.
Querying MAST for TIC 234349833 (M3.5V)...
  --> No 20-second SPOC data found
Querying MAST for TIC 234295610 (M4V)...
  --> Success: Extracted 15 qualified flares.
Querying MAST for TIC 177309964 (M4.5V)...
  --> Success: Extracted 45 qualified flares.
Querying MAST for TIC 441162396 (M5V)...
  --> No 20-second SPOC data found
Querying MAST for TIC 231541244 (M6V)...
  --> No 20-second SPOC data found
Querying